# Vaelo — One Coherent Demo Client Across All Three Pipelines

**Subject company: Meridian Auto Components Pvt Ltd** (Coimbatore, Tamil Nadu) — an auto-ancillary manufacturer riding the EV supply-chain shift.

This notebook runs all three Vaelo pipelines against **the same company, the same financial history**, so the output is one coherent CA-ready demo packet instead of three disconnected examples:

1. **Valuation Report** — standalone DCF (trigger: succession sale)
2. **Deal Feasibility Report** — Horizon Industrials Pvt Ltd considers acquiring Meridian. The target's `standalone_value` is pulled **directly from Pipeline 1's DCF equity value** — not re-estimated. This is the connective tissue between the two pipelines.
3. **Financial Health Snapshot** — Meridian's own health check, run on the **identical 5-year revenue history** used in the valuation.

A consistency check at the end asserts all three reports actually agree with each other on the shared numbers — not just that each one runs standalone.

The class/function definitions below are copied unchanged from the three individual pipeline notebooks — only the example data at the bottom of each is replaced with the shared Meridian scenario.

## 1. Pipeline Definitions

### 1.1 Valuation Report engine (Pipeline 1)

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional
from datetime import date

@dataclass
class ClientMeta:
    """Who this report is for and why — SRS FR-1.1"""
    client_name: str
    ca_firm_name: str
    trigger_event: str          # 'loan', 'sale', 'merger', 'investor_pitch', 'periodic'
    report_date: str = field(default_factory=lambda: date.today().isoformat())
    sector: str = "General"


@dataclass
class FinancialInputs:
    """Raw financial data submitted for the client — SRS FR-1.2"""
    historical_revenue: List[float]      # most recent years, oldest to newest, in Cr
    historical_ebitda: List[float]
    current_ebit: float
    current_da: float                     # depreciation & amortization
    current_capex: float
    current_nwc_change: float             # change in net working capital
    tax_rate: float                       # e.g. 0.25 for 25%
    net_debt: float                       # total debt - cash & equivalents


@dataclass
class WACCInputs:
    """Build-up method inputs — no public beta exists for a private SME,
    so CAPM alone doesn't apply. See the earlier guide for sourcing each figure."""
    risk_free_rate: float
    equity_risk_premium: float
    size_premium: float
    company_specific_premium: float
    cost_of_debt: float
    equity_weight: float                  # E / (E+D)
    debt_weight: float                    # D / (E+D)
    tax_rate: float

    def cost_of_equity(self) -> float:
        return (
            self.risk_free_rate
            + self.equity_risk_premium
            + self.size_premium
            + self.company_specific_premium
        )

    def wacc(self) -> float:
        ke = self.cost_of_equity()
        after_tax_kd = self.cost_of_debt * (1 - self.tax_rate)
        return (self.equity_weight * ke) + (self.debt_weight * after_tax_kd)


@dataclass
class ProjectionAssumptions:
    """Forward-looking assumptions — CA-adjustable, SRS FR-2.4"""
    projection_years: int
    revenue_growth_rates: List[float]      # one per projection year
    ebitda_margin: float
    da_as_pct_revenue: float
    capex_as_pct_revenue: float
    nwc_change_as_pct_revenue: float
    terminal_growth_rate: float            # must be < WACC


@dataclass
class ComparableAssumptions:
    """For the cross-check in Section 4 — sourced per-sector (e.g. Damodaran data,
    Indian sector reports). This is a sanity check, not the primary valuation method."""
    ev_ebitda_multiple_low: float
    ev_ebitda_multiple_high: float


@dataclass
class ValuationRequest:
    """The single object a CA submits — the intake boundary for this pipeline."""
    meta: ClientMeta
    financials: FinancialInputs
    wacc_inputs: WACCInputs
    assumptions: ProjectionAssumptions
    comparables: ComparableAssumptions

def project_free_cash_flows(base_revenue: float, req: ValuationRequest) -> List[float]:
    """Project FCF for each year of the projection period."""
    fcfs = []
    revenue = base_revenue
    inputs = req.financials
    assumptions = req.assumptions

    for growth_rate in assumptions.revenue_growth_rates:
        revenue = revenue * (1 + growth_rate)
        ebitda = revenue * assumptions.ebitda_margin
        da = revenue * assumptions.da_as_pct_revenue
        ebit = ebitda - da
        capex = revenue * assumptions.capex_as_pct_revenue
        nwc_change = revenue * assumptions.nwc_change_as_pct_revenue

        fcf = ebit * (1 - inputs.tax_rate) + da - capex - nwc_change
        fcfs.append(fcf)

    return fcfs


def discount_cash_flows(fcfs: List[float], wacc: float) -> List[float]:
    """Return present value of each projected FCF."""
    return [fcf / ((1 + wacc) ** (t + 1)) for t, fcf in enumerate(fcfs)]


def terminal_value(final_year_fcf: float, wacc: float, terminal_growth: float) -> float:
    """Gordon Growth terminal value. Hard constraint: g must be < WACC."""
    if terminal_growth >= wacc:
        raise ValueError(
            f"Terminal growth rate ({terminal_growth:.1%}) must be less than "
            f"WACC ({wacc:.1%}) — a terminal growth rate above the discount rate "
            f"implies infinite value, which is not a valid result."
        )
    return final_year_fcf * (1 + terminal_growth) / (wacc - terminal_growth)


def calculate_enterprise_value(base_revenue: float, req: ValuationRequest) -> dict:
    """The full DCF calculation. Returns every intermediate figure, not just
    the final number — auditability is the entire point of this pipeline."""

    wacc = req.wacc_inputs.wacc()

    fcfs = project_free_cash_flows(base_revenue, req)
    pv_fcfs = discount_cash_flows(fcfs, wacc)

    tv = terminal_value(fcfs[-1], wacc, req.assumptions.terminal_growth_rate)
    pv_tv = tv / ((1 + wacc) ** req.assumptions.projection_years)

    enterprise_value = sum(pv_fcfs) + pv_tv
    equity_value = enterprise_value - req.financials.net_debt

    return {
        "wacc": wacc,
        "cost_of_equity": req.wacc_inputs.cost_of_equity(),
        "projected_fcfs": fcfs,
        "present_value_fcfs": pv_fcfs,
        "terminal_value": tv,
        "pv_terminal_value": pv_tv,
        "enterprise_value": enterprise_value,
        "net_debt": req.financials.net_debt,
        "equity_value": equity_value,
    }


def sensitivity_grid(
    base_revenue: float,
    req: ValuationRequest,
    wacc_range: List[float],
    growth_range: List[float],
) -> dict:
    """Build a WACC x terminal-growth sensitivity matrix — SRS FR-2.2.
    Shows how the valuation moves as assumptions change, rather than
    presenting a single number as gospel."""

    grid = {}
    for w in wacc_range:
        row = {}
        for g in growth_range:
            fcfs = project_free_cash_flows(base_revenue, req)
            pv_fcfs = discount_cash_flows(fcfs, w)
            tv = terminal_value(fcfs[-1], w, g)
            pv_tv = tv / ((1 + w) ** req.assumptions.projection_years)
            ev = sum(pv_fcfs) + pv_tv
            row[f"g={g:.1%}"] = round(ev, 2)
        grid[f"WACC={w:.1%}"] = row

    return grid

def comparable_cross_check(latest_ebitda: float, req: ValuationRequest, dcf_ev: float) -> dict:
    low = latest_ebitda * req.comparables.ev_ebitda_multiple_low
    high = latest_ebitda * req.comparables.ev_ebitda_multiple_high
    midpoint = (low + high) / 2

    deviation = abs(dcf_ev - midpoint) / midpoint if midpoint else 0
    flag = deviation > 0.35

    return {
        "multiple_low_ev": round(low, 2),
        "multiple_high_ev": round(high, 2),
        "multiple_midpoint_ev": round(midpoint, 2),
        "dcf_vs_multiple_deviation_pct": round(deviation * 100, 1),
        "flagged_for_review": flag,
    }

def build_valuation_report(
    req: ValuationRequest,
    dcf_result: dict,
    sensitivity: dict,
    cross_check: dict,
) -> str:
    """Templated report generation — plain string formatting, no LLM."""

    lines = []
    lines.append(f"VALUATION REPORT — {req.meta.client_name}")
    lines.append(f"Prepared for: {req.meta.ca_firm_name}")
    lines.append(f"Date: {req.meta.report_date}")
    lines.append(f"Trigger event: {req.meta.trigger_event}")
    lines.append(f"Sector: {req.meta.sector}")
    lines.append("=" * 60)

    lines.append("\n--- DISCOUNT RATE (WACC) BUILD-UP ---")
    lines.append(f"Cost of Equity:       {dcf_result['cost_of_equity']:.2%}")
    lines.append(f"WACC:                 {dcf_result['wacc']:.2%}")

    lines.append("\n--- PROJECTED FREE CASH FLOWS (Rs. Cr) ---")
    for i, (fcf, pv) in enumerate(zip(dcf_result["projected_fcfs"], dcf_result["present_value_fcfs"]), start=1):
        lines.append(f"Year {i}: FCF = {fcf:.2f}  |  Present Value = {pv:.2f}")

    lines.append("\n--- TERMINAL VALUE ---")
    lines.append(f"Terminal Value:       Rs. {dcf_result['terminal_value']:.2f} Cr")
    lines.append(f"PV of Terminal Value: Rs. {dcf_result['pv_terminal_value']:.2f} Cr")

    lines.append("\n--- VALUATION SUMMARY ---")
    lines.append(f"Enterprise Value:     Rs. {dcf_result['enterprise_value']:.2f} Cr")
    lines.append(f"Less: Net Debt:       Rs. {dcf_result['net_debt']:.2f} Cr")
    lines.append(f"Equity Value:         Rs. {dcf_result['equity_value']:.2f} Cr")

    lines.append("\n--- COMPARABLE-MULTIPLE CROSS-CHECK ---")
    lines.append(f"Implied EV range (multiple method): Rs. {cross_check['multiple_low_ev']:.2f} Cr - Rs. {cross_check['multiple_high_ev']:.2f} Cr")
    lines.append(f"DCF vs. multiple deviation:          {cross_check['dcf_vs_multiple_deviation_pct']:.1f}%")
    if cross_check["flagged_for_review"]:
        lines.append("FLAG: DCF and multiple-based estimates diverge by more than 35%.")
        lines.append("      Review assumptions before presenting this report to the client.")
    else:
        lines.append("DCF and multiple-based estimates are broadly consistent.")

    lines.append("\n--- SENSITIVITY ANALYSIS (Enterprise Value, Rs. Cr) ---")
    header = "WACC \\ g".ljust(12) + "".join(
        g.ljust(12) for g in list(sensitivity.values())[0].keys()
    )
    lines.append(header)
    for wacc_label, row in sensitivity.items():
        line = wacc_label.ljust(12) + "".join(str(v).ljust(12) for v in row.values())
        lines.append(line)

    lines.append("\n" + "=" * 60)
    lines.append("This report is formula-driven and fully auditable. Every figure above")
    lines.append("traces to the assumptions and inputs supplied for this engagement.")
    lines.append("Prepared via Vaelo — for review by the submitting CA before client delivery.")

    return "\n".join(lines)

### 1.2 Deal Feasibility engine (Pipeline 2)

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional
from datetime import date

@dataclass
class DealMeta:
    """Who this report is for and why."""
    acquirer_name: str
    target_name: str
    ca_firm_name: str
    deal_rationale: str          # e.g. 'market expansion', 'vertical integration', 'succession sale'
    report_date: str = field(default_factory=lambda: date.today().isoformat())
    sector: str = "General"


@dataclass
class CompanyProfile:
    """Standalone financials for either the acquirer or the target, in Rs Cr
    unless noted. `standalone_value` and `fallback_ev_ebitda_multiple` are only
    used for the target (premium analysis)."""
    name: str
    revenue: float
    ebitda: float
    net_income: float
    shares_outstanding: float          # absolute share count
    net_debt: float                    # total debt - cash & equivalents
    standalone_value: Optional[float] = None          # from Pipeline 1 DCF, if available
    fallback_ev_ebitda_multiple: float = 6.0           # used only if standalone_value is None


@dataclass
class DealTerms:
    """The proposed deal being tested for feasibility."""
    deal_type: str                     # 'acquisition' or 'merger'
    purchase_price: float              # Rs Cr, agreed equity value for target
    cash_component_pct: float          # 0-1
    stock_component_pct: float         # 0-1, cash_component_pct + stock_component_pct = 1
    acquirer_share_price: float        # Rs, needed to size stock issuance


@dataclass
class FinancingAssumptions:
    """How the cash portion of the deal gets funded."""
    new_debt_raised: float             # Rs Cr
    cost_of_new_debt: float            # interest rate on the new debt
    acquirer_cash_used: float          # Rs Cr drawn from acquirer's own reserves
    tax_rate: float


@dataclass
class SynergyAssumptions:
    """Run-rate synergy assumptions and how fast they ramp in."""
    annual_cost_synergies: float           # Rs Cr/year at full run-rate
    annual_revenue_synergies: float        # Rs Cr/year at full run-rate
    synergy_ebitda_margin: float           # margin applied to revenue synergies for EBITDA impact
    ramp_up_years: int                     # years to reach full run-rate, linear ramp
    synergy_discount_rate: float           # for NPV — typically the combined entity's WACC


@dataclass
class DealFeasibilityRequest:
    """The single object a CA submits — the intake boundary for this pipeline."""
    meta: DealMeta
    acquirer: CompanyProfile
    target: CompanyProfile
    deal_terms: DealTerms
    financing: FinancingAssumptions
    synergies: SynergyAssumptions

def premium_analysis(target: CompanyProfile, deal_terms: DealTerms) -> dict:
    """Compare the agreed purchase price to the target's standalone value."""
    standalone_value = (
        target.standalone_value
        if target.standalone_value is not None
        else target.ebitda * target.fallback_ev_ebitda_multiple
    )
    premium_pct = (
        ((deal_terms.purchase_price - standalone_value) / standalone_value) * 100
        if standalone_value else 0
    )
    return {
        "target_standalone_value": round(standalone_value, 2),
        "purchase_price": deal_terms.purchase_price,
        "premium_pct": round(premium_pct, 1),
    }


def sources_and_uses(deal_terms: DealTerms, financing: FinancingAssumptions, purchase_price: float) -> dict:
    """Check that the financing plan actually funds the deal. Advisory/transaction
    fees are excluded for simplicity — add a fees line if you need precision here."""
    cash_needed = purchase_price * deal_terms.cash_component_pct
    stock_needed = purchase_price * deal_terms.stock_component_pct
    cash_sources = financing.acquirer_cash_used + financing.new_debt_raised
    cash_funding_gap = round(cash_needed - cash_sources, 2)
    # stock_needed is in Rs Cr; acquirer_share_price is in plain Rs — convert before dividing
    new_shares_issued = (
        (stock_needed * 1e7) / deal_terms.acquirer_share_price
        if deal_terms.acquirer_share_price else 0
    )

    return {
        "cash_needed": round(cash_needed, 2),
        "stock_needed": round(stock_needed, 2),
        "cash_sources": round(cash_sources, 2),
        "cash_funding_gap": cash_funding_gap,
        "new_shares_issued": round(new_shares_issued, 0),
    }

def pro_forma_combined(acquirer: CompanyProfile, target: CompanyProfile, financing: FinancingAssumptions) -> dict:
    """Combine the two standalone P&Ls and layer on the interest cost of new debt.
    Pre-synergy — this is the day-one picture before any operational integration."""
    combined_revenue = acquirer.revenue + target.revenue
    combined_ebitda_pre_synergy = acquirer.ebitda + target.ebitda

    additional_interest_pretax = financing.new_debt_raised * financing.cost_of_new_debt
    additional_interest_after_tax = additional_interest_pretax * (1 - financing.tax_rate)

    combined_net_income_pre_synergy = (
        acquirer.net_income + target.net_income - additional_interest_after_tax
    )

    return {
        "combined_revenue": round(combined_revenue, 2),
        "combined_ebitda_pre_synergy": round(combined_ebitda_pre_synergy, 2),
        "additional_interest_after_tax": round(additional_interest_after_tax, 2),
        "combined_net_income_pre_synergy": round(combined_net_income_pre_synergy, 2),
    }


def synergy_npv(synergies: SynergyAssumptions, tax_rate: float) -> dict:
    """NPV of synergies: linear ramp to full run-rate over `ramp_up_years`, then a
    flat (no-growth) perpetuity from full run-rate — deliberately conservative,
    doesn't assume synergies keep growing forever."""
    full_run_rate_pretax = (
        synergies.annual_cost_synergies
        + synergies.annual_revenue_synergies * synergies.synergy_ebitda_margin
    )
    full_run_rate_after_tax = full_run_rate_pretax * (1 - tax_rate)

    ramp_cashflows = [
        full_run_rate_after_tax * (yr / synergies.ramp_up_years)
        for yr in range(1, synergies.ramp_up_years + 1)
    ]
    pv_ramp = sum(
        cf / ((1 + synergies.synergy_discount_rate) ** t)
        for t, cf in enumerate(ramp_cashflows, start=1)
    )

    perpetuity_value = full_run_rate_after_tax / synergies.synergy_discount_rate
    pv_perpetuity = perpetuity_value / ((1 + synergies.synergy_discount_rate) ** synergies.ramp_up_years)

    total_synergy_npv = pv_ramp + pv_perpetuity

    return {
        "full_run_rate_pretax": round(full_run_rate_pretax, 2),
        "full_run_rate_after_tax": round(full_run_rate_after_tax, 2),
        "pv_ramp_period": round(pv_ramp, 2),
        "pv_terminal_perpetuity": round(pv_perpetuity, 2),
        "total_synergy_npv": round(total_synergy_npv, 2),
    }

def accretion_dilution(acquirer: CompanyProfile, pro_forma: dict, su: dict, synergy: dict) -> dict:
    """Does the deal help or hurt the acquirer's existing shareholders, on an EPS
    basis? Shown both pre-synergy (day one) and at full synergy run-rate."""
    acquirer_standalone_eps = (acquirer.net_income * 1e7) / acquirer.shares_outstanding
    pro_forma_shares = acquirer.shares_outstanding + su["new_shares_issued"]

    pro_forma_eps_pre_synergy = (pro_forma["combined_net_income_pre_synergy"] * 1e7) / pro_forma_shares
    change_pre_synergy_pct = (pro_forma_eps_pre_synergy - acquirer_standalone_eps) / acquirer_standalone_eps

    net_income_post_synergy = pro_forma["combined_net_income_pre_synergy"] + synergy["full_run_rate_after_tax"]
    pro_forma_eps_post_synergy = (net_income_post_synergy * 1e7) / pro_forma_shares
    change_post_synergy_pct = (pro_forma_eps_post_synergy - acquirer_standalone_eps) / acquirer_standalone_eps

    return {
        "acquirer_standalone_eps": round(acquirer_standalone_eps, 2),
        "pro_forma_shares": round(pro_forma_shares, 0),
        "pro_forma_eps_pre_synergy": round(pro_forma_eps_pre_synergy, 2),
        "change_pre_synergy_pct": round(change_pre_synergy_pct * 100, 1),
        "pro_forma_eps_post_synergy_full_run_rate": round(pro_forma_eps_post_synergy, 2),
        "change_post_synergy_pct": round(change_post_synergy_pct * 100, 1),
    }


def leverage_feasibility(acquirer: CompanyProfile, target: CompanyProfile,
                          financing: FinancingAssumptions, pro_forma: dict,
                          threshold: float = 4.0) -> dict:
    """Pro-forma Net Debt / EBITDA against a feasibility threshold. 4.0x is a
    reasonable default ceiling for SME lending covenants — adjust per the CA's
    actual banking relationships if they have a tighter or looser covenant."""
    pro_forma_net_debt = (
        acquirer.net_debt + target.net_debt
        + financing.new_debt_raised + financing.acquirer_cash_used
    )
    combined_ebitda = pro_forma["combined_ebitda_pre_synergy"]
    leverage_ratio = pro_forma_net_debt / combined_ebitda if combined_ebitda else float("inf")

    return {
        "pro_forma_net_debt": round(pro_forma_net_debt, 2),
        "leverage_ratio": round(leverage_ratio, 2),
        "leverage_threshold": threshold,
        "leverage_flagged": leverage_ratio > threshold,
    }


def breakeven_synergy_required(accretion: dict, pro_forma: dict, financing: FinancingAssumptions) -> dict:
    """If the deal is dilutive pre-synergy, how much after-tax synergy is needed
    just to get back to EPS-neutral? Zero if the deal is already accretive."""
    if accretion["change_pre_synergy_pct"] >= 0:
        return {
            "breakeven_after_tax_synergy_required": 0.0,
            "breakeven_pretax_synergy_required": 0.0,
            "note": "Deal is already EPS-neutral or accretive before synergies.",
        }

    required_net_income = accretion["acquirer_standalone_eps"] * accretion["pro_forma_shares"] / 1e7
    current_net_income = pro_forma["combined_net_income_pre_synergy"]
    required_after_tax = required_net_income - current_net_income
    required_pretax = required_after_tax / (1 - financing.tax_rate)

    return {
        "breakeven_after_tax_synergy_required": round(required_after_tax, 2),
        "breakeven_pretax_synergy_required": round(required_pretax, 2),
    }


def overall_feasibility_verdict(premium: dict, leverage: dict, accretion: dict,
                                 synergy: dict, breakeven: dict) -> dict:
    """Rule-based verdict, not a judgment call — every reason traces to a specific
    flag computed above. Thresholds (50% premium, -10% dilution) are defaults;
    tune them per deal type or the CA's own risk appetite."""
    reasons = []

    if leverage["leverage_flagged"]:
        reasons.append(
            f"Pro-forma leverage of {leverage['leverage_ratio']}x exceeds the "
            f"{leverage['leverage_threshold']}x threshold."
        )
    if premium["premium_pct"] > 50:
        reasons.append(
            f"Purchase price implies a {premium['premium_pct']}% premium to standalone "
            f"value — unusually high, re-examine assumptions."
        )
    if (accretion["change_pre_synergy_pct"] < -10
            and synergy["total_synergy_npv"] < breakeven.get("breakeven_after_tax_synergy_required", 0) * 3):
        reasons.append(
            "Deal is meaningfully dilutive pre-synergy and synergy NPV does not "
            "comfortably cover the breakeven requirement."
        )

    if not reasons:
        verdict = "FEASIBLE AS STRUCTURED"
    elif len(reasons) == 1:
        verdict = "FEASIBLE WITH CONDITIONS — review flagged item below"
    else:
        verdict = "NOT RECOMMENDED AS STRUCTURED — multiple flags raised"

    return {"verdict": verdict, "reasons": reasons}

def calculate_deal_feasibility(req: DealFeasibilityRequest) -> dict:
    """Runs the full deterministic feasibility check end to end."""
    premium = premium_analysis(req.target, req.deal_terms)
    su = sources_and_uses(req.deal_terms, req.financing, premium["purchase_price"])
    pro_forma = pro_forma_combined(req.acquirer, req.target, req.financing)
    synergy = synergy_npv(req.synergies, req.financing.tax_rate)
    accretion = accretion_dilution(req.acquirer, pro_forma, su, synergy)
    leverage = leverage_feasibility(req.acquirer, req.target, req.financing, pro_forma)
    breakeven = breakeven_synergy_required(accretion, pro_forma, req.financing)
    verdict = overall_feasibility_verdict(premium, leverage, accretion, synergy, breakeven)

    return {
        "premium": premium,
        "sources_and_uses": su,
        "pro_forma": pro_forma,
        "synergy": synergy,
        "accretion_dilution": accretion,
        "leverage": leverage,
        "breakeven": breakeven,
        "verdict": verdict,
    }

def build_deal_feasibility_report(req: DealFeasibilityRequest, result: dict) -> str:
    """Templated report generation — plain string formatting, no LLM."""
    lines = []
    lines.append(f"DEAL FEASIBILITY REPORT — {req.meta.acquirer_name} / {req.meta.target_name}")
    lines.append(f"Prepared for: {req.meta.ca_firm_name}")
    lines.append(f"Date: {req.meta.report_date}")
    lines.append(f"Deal type: {req.deal_terms.deal_type}")
    lines.append(f"Rationale: {req.meta.deal_rationale}")
    lines.append(f"Sector: {req.meta.sector}")
    lines.append("=" * 60)

    p = result["premium"]
    lines.append("\n--- PREMIUM ANALYSIS ---")
    lines.append(f"Target standalone value: Rs. {p['target_standalone_value']:.2f} Cr")
    lines.append(f"Agreed purchase price:   Rs. {p['purchase_price']:.2f} Cr")
    lines.append(f"Premium to standalone value: {p['premium_pct']:.1f}%")

    su = result["sources_and_uses"]
    lines.append("\n--- SOURCES & USES ---")
    lines.append(f"Cash required:   Rs. {su['cash_needed']:.2f} Cr")
    lines.append(f"Stock required:  Rs. {su['stock_needed']:.2f} Cr  (new shares issued: {su['new_shares_issued']:.0f})")
    lines.append(f"Cash sources (acquirer cash + new debt): Rs. {su['cash_sources']:.2f} Cr")
    if abs(su["cash_funding_gap"]) > 0.01:
        lines.append(f"FLAG: Funding gap of Rs. {su['cash_funding_gap']:.2f} Cr — sources do not match cash required.")
    else:
        lines.append("Sources and uses balance.")

    pf = result["pro_forma"]
    lines.append("\n--- PRO-FORMA COMBINED ENTITY (pre-synergy) ---")
    lines.append(f"Combined Revenue: Rs. {pf['combined_revenue']:.2f} Cr")
    lines.append(f"Combined EBITDA:  Rs. {pf['combined_ebitda_pre_synergy']:.2f} Cr")
    lines.append(f"Combined Net Income (after new-debt interest): Rs. {pf['combined_net_income_pre_synergy']:.2f} Cr")

    syn = result["synergy"]
    lines.append("\n--- SYNERGY NPV ---")
    lines.append(f"Full run-rate synergies (after-tax): Rs. {syn['full_run_rate_after_tax']:.2f} Cr/year")
    lines.append(f"PV during ramp-up period:  Rs. {syn['pv_ramp_period']:.2f} Cr")
    lines.append(f"PV of terminal perpetuity: Rs. {syn['pv_terminal_perpetuity']:.2f} Cr")
    lines.append(f"Total Synergy NPV:         Rs. {syn['total_synergy_npv']:.2f} Cr")

    ad = result["accretion_dilution"]
    lines.append("\n--- ACCRETION / DILUTION ---")
    lines.append(f"Acquirer standalone EPS: Rs. {ad['acquirer_standalone_eps']:.2f}")
    lines.append(f"Pro-forma EPS (pre-synergy):            Rs. {ad['pro_forma_eps_pre_synergy']:.2f}  ({ad['change_pre_synergy_pct']:+.1f}%)")
    lines.append(f"Pro-forma EPS (full run-rate synergy):  Rs. {ad['pro_forma_eps_post_synergy_full_run_rate']:.2f}  ({ad['change_post_synergy_pct']:+.1f}%)")

    lev = result["leverage"]
    lines.append("\n--- LEVERAGE & FINANCING FEASIBILITY ---")
    lines.append(f"Pro-forma Net Debt: Rs. {lev['pro_forma_net_debt']:.2f} Cr")
    lines.append(f"Pro-forma Net Debt / EBITDA: {lev['leverage_ratio']:.2f}x  (threshold: {lev['leverage_threshold']:.1f}x)")
    if lev["leverage_flagged"]:
        lines.append("FLAG: Pro-forma leverage exceeds the feasibility threshold.")
    else:
        lines.append("Pro-forma leverage is within the feasibility threshold.")

    be = result["breakeven"]
    lines.append("\n--- BREAKEVEN SYNERGY REQUIRED ---")
    if be.get("breakeven_after_tax_synergy_required", 0) > 0:
        lines.append(f"After-tax synergy needed for EPS-neutrality: Rs. {be['breakeven_after_tax_synergy_required']:.2f} Cr/year")
        lines.append(f"Pre-tax equivalent: Rs. {be['breakeven_pretax_synergy_required']:.2f} Cr/year")
    else:
        lines.append(be.get("note", "Deal is already EPS-neutral or accretive before synergies."))

    v = result["verdict"]
    lines.append("\n--- OVERALL FEASIBILITY VERDICT ---")
    lines.append(v["verdict"])
    for r in v["reasons"]:
        lines.append(f"  - {r}")

    lines.append("\n" + "=" * 60)
    lines.append("This report is formula-driven and fully auditable. Every figure above")
    lines.append("traces to the assumptions and inputs supplied for this engagement.")
    lines.append("Prepared via Vaelo — for review by the submitting CA before client delivery.")

    return "\n".join(lines)

### 1.3 Financial Health Snapshot engine (Pipeline 3)

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional
from datetime import date
import statistics

@dataclass
class SnapshotMeta:
    """Who this snapshot is for and why."""
    client_name: str
    ca_firm_name: str
    report_date: str = field(default_factory=lambda: date.today().isoformat())
    sector: str = "General"


@dataclass
class LiquidityInputs:
    """Latest balance sheet position, Rs Cr."""
    current_assets: float
    current_liabilities: float


@dataclass
class ExpenseGrowthInputs:
    """Most recent two full years, Rs Cr. Oldest to newest."""
    revenue_prior_year: float
    revenue_current_year: float
    operating_expenses_prior_year: float
    operating_expenses_current_year: float


@dataclass
class CashRunwayInputs:
    """Rs Cr. If the business is net cash-generative (not burning cash),
    set monthly_net_cash_flow >= 0 and the engine reports runway as N/A —
    a business that generates cash doesn't have a 'burn runway' to measure."""
    cash_and_equivalents: float
    monthly_net_cash_flow: float          # negative if burning cash, positive if generating


@dataclass
class RevenueVolatilityInputs:
    """3+ years of historical annual revenue, Rs Cr, oldest to newest.
    More years gives a more reliable volatility read — 3 is the practical minimum."""
    historical_revenue: List[float]


@dataclass
class HealthSnapshotRequest:
    """The single object a CA submits — the intake boundary for this pipeline."""
    meta: SnapshotMeta
    liquidity: LiquidityInputs
    expense_growth: ExpenseGrowthInputs
    cash_runway: CashRunwayInputs
    revenue_volatility: RevenueVolatilityInputs

def score_liquidity(inputs: LiquidityInputs) -> dict:
    """Current Ratio -> 0-100 score with a plain-language label."""
    if inputs.current_liabilities <= 0:
        current_ratio = float("inf")
    else:
        current_ratio = inputs.current_assets / inputs.current_liabilities

    if current_ratio < 1.0:
        score = max(0, current_ratio * 40)                      # 0.0 -> 0, 1.0 -> 40
        label = "Weak"
        note = "Current liabilities exceed current assets — a real short-term liquidity risk."
    elif current_ratio < 1.5:
        score = 40 + (current_ratio - 1.0) / 0.5 * 25            # 1.0 -> 40, 1.5 -> 65
        label = "Adequate"
        note = "Liquidity is workable but has limited cushion."
    elif current_ratio <= 3.0:
        score = 65 + (current_ratio - 1.5) / 1.5 * 25            # 1.5 -> 65, 3.0 -> 90
        label = "Strong"
        note = "Healthy short-term liquidity cushion."
    else:
        score = 85                                                # capped, not climbing further
        label = "Excess (possible idle capital)"
        note = "Very high current ratio — worth checking whether working capital is being deployed efficiently rather than sitting idle."

    return {
        "current_ratio": round(current_ratio, 2) if current_ratio != float("inf") else None,
        "score": round(min(max(score, 0), 100), 1),
        "label": label,
        "note": note,
    }

def score_expense_growth(inputs: ExpenseGrowthInputs) -> dict:
    """Expense growth rate vs. revenue growth rate -> 0-100 score."""
    revenue_growth = (
        (inputs.revenue_current_year - inputs.revenue_prior_year) / inputs.revenue_prior_year
        if inputs.revenue_prior_year else 0
    )
    expense_growth = (
        (inputs.operating_expenses_current_year - inputs.operating_expenses_prior_year)
        / inputs.operating_expenses_prior_year
        if inputs.operating_expenses_prior_year else 0
    )
    spread = expense_growth - revenue_growth   # negative = expenses growing slower than revenue (good)

    if spread <= -0.05:
        score = 100
        label = "Excellent"
        note = "Expenses are growing meaningfully slower than revenue — operating leverage is improving."
    elif spread <= 0:
        score = 75 + (abs(spread) / 0.05) * 25 if spread != 0 else 75   # 0 -> 75, -0.05 -> 100
        label = "Good"
        note = "Expenses are keeping pace with or growing slightly slower than revenue."
    elif spread <= 0.05:
        score = 40 + (1 - spread / 0.05) * 35                           # 0 -> 75, 0.05 -> 40
        label = "Caution"
        note = "Expenses are growing modestly faster than revenue — worth monitoring."
    else:
        score = max(0, 40 - (spread - 0.05) * 200)                      # steep drop past 5pt spread
        label = "Concerning"
        note = "Expenses are significantly outgrowing revenue — margin compression risk."

    return {
        "revenue_growth_pct": round(revenue_growth * 100, 1),
        "expense_growth_pct": round(expense_growth * 100, 1),
        "spread_pct": round(spread * 100, 1),
        "score": round(min(max(score, 0), 100), 1),
        "label": label,
        "note": note,
    }

def score_cash_runway(inputs: CashRunwayInputs) -> dict:
    """Months of runway at current burn rate -> 0-100 score."""
    if inputs.monthly_net_cash_flow >= 0:
        return {
            "runway_months": None,
            "score": 100.0,
            "label": "Cash-generative",
            "note": "Business is generating positive net cash flow — no burn runway to measure.",
        }

    monthly_burn = abs(inputs.monthly_net_cash_flow)
    runway_months = inputs.cash_and_equivalents / monthly_burn if monthly_burn else float("inf")

    if runway_months < 3:
        score = runway_months / 3 * 20                    # 0 -> 0, 3 -> 20
        label = "Critical"
        note = "Less than 3 months of runway at current burn rate — immediate attention required."
    elif runway_months < 6:
        score = 20 + (runway_months - 3) / 3 * 30           # 3 -> 20, 6 -> 50
        label = "Weak"
        note = "Runway is short — limited room to absorb a slow quarter or delayed receivables."
    elif runway_months < 12:
        score = 50 + (runway_months - 6) / 6 * 25           # 6 -> 50, 12 -> 75
        label = "Adequate"
        note = "Reasonable buffer, though worth monitoring if burn increases."
    else:
        score = min(75 + (runway_months - 12) / 6 * 25, 100)  # 12 -> 75, 18+ -> 100
        label = "Strong"
        note = "Healthy cash buffer relative to current burn rate."

    return {
        "runway_months": round(runway_months, 1),
        "score": round(min(max(score, 0), 100), 1),
        "label": label,
        "note": note,
    }

def score_revenue_volatility(inputs: RevenueVolatilityInputs) -> dict:
    """Coefficient of variation of historical revenue -> 0-100 score."""
    revenues = inputs.historical_revenue
    if len(revenues) < 3:
        raise ValueError(
            f"Revenue volatility requires at least 3 years of historical revenue, "
            f"got {len(revenues)} — this is a hard minimum for a meaningful read, not a suggestion."
        )

    mean_rev = statistics.mean(revenues)
    stdev_rev = statistics.stdev(revenues)
    cv = stdev_rev / mean_rev if mean_rev else float("inf")

    if cv < 0.05:
        score = 100 - (cv / 0.05) * 10                      # 0 -> 100, 0.05 -> 90
        label = "Very stable"
        note = "Revenue has been highly predictable year over year."
    elif cv < 0.15:
        score = 90 - (cv - 0.05) / 0.10 * 20                 # 0.05 -> 90, 0.15 -> 70
        label = "Stable"
        note = "Revenue shows normal, manageable year-to-year variation."
    elif cv < 0.30:
        score = 70 - (cv - 0.15) / 0.15 * 30                 # 0.15 -> 70, 0.30 -> 40
        label = "Variable"
        note = "Revenue swings meaningfully year to year — worth understanding the cause (seasonality, customer concentration, etc.)."
    else:
        score = max(0, 40 - (cv - 0.30) * 100)
        label = "Highly volatile"
        note = "Revenue is highly unpredictable — this materially increases risk independent of the average growth rate."

    return {
        "coefficient_of_variation": round(cv, 3),
        "mean_revenue": round(mean_rev, 2),
        "score": round(min(max(score, 0), 100), 1),
        "label": label,
        "note": note,
    }

def calculate_health_snapshot(req: HealthSnapshotRequest) -> dict:
    """Runs all four independent sub-scores. No composite/weighted score by design."""
    liquidity = score_liquidity(req.liquidity)
    expense_growth = score_expense_growth(req.expense_growth)
    cash_runway = score_cash_runway(req.cash_runway)
    revenue_volatility = score_revenue_volatility(req.revenue_volatility)

    # Flag any sub-score below a "worth watching" threshold — this is a highlight,
    # not a collapsed composite number.
    flags = []
    for name, result in [
        ("Liquidity", liquidity),
        ("Expense Growth", expense_growth),
        ("Cash Runway", cash_runway),
        ("Revenue Volatility", revenue_volatility),
    ]:
        if result["score"] < 40:
            flags.append(f"{name}: {result['label']} ({result['score']}/100) — {result['note']}")

    return {
        "liquidity": liquidity,
        "expense_growth": expense_growth,
        "cash_runway": cash_runway,
        "revenue_volatility": revenue_volatility,
        "flags": flags,
    }

def build_health_snapshot_report(req: HealthSnapshotRequest, result: dict) -> str:
    """Templated report generation — plain string formatting, no LLM."""
    lines = []
    lines.append(f"FINANCIAL HEALTH SNAPSHOT — {req.meta.client_name}")
    lines.append(f"Prepared for: {req.meta.ca_firm_name}")
    lines.append(f"Date: {req.meta.report_date}")
    lines.append(f"Sector: {req.meta.sector}")
    lines.append("=" * 60)
    lines.append("This snapshot presents four independent measures of financial health.")
    lines.append("They are shown separately, not collapsed into a single score, so each")
    lines.append("dimension can be reviewed and explained on its own.")

    liq = result["liquidity"]
    lines.append("\n--- 1. LIQUIDITY (Current Ratio) ---")
    lines.append(f"Current Ratio: {liq['current_ratio']}")
    lines.append(f"Score: {liq['score']}/100  ({liq['label']})")
    lines.append(f"Note: {liq['note']}")

    exp = result["expense_growth"]
    lines.append("\n--- 2. EXPENSE GROWTH RATE ---")
    lines.append(f"Revenue growth: {exp['revenue_growth_pct']:+.1f}%   Expense growth: {exp['expense_growth_pct']:+.1f}%")
    lines.append(f"Spread (expense - revenue growth): {exp['spread_pct']:+.1f} pts")
    lines.append(f"Score: {exp['score']}/100  ({exp['label']})")
    lines.append(f"Note: {exp['note']}")

    cash = result["cash_runway"]
    lines.append("\n--- 3. CASH RUNWAY ---")
    if cash["runway_months"] is None:
        lines.append("Runway: N/A (cash-generative)")
    else:
        lines.append(f"Runway: {cash['runway_months']} months")
    lines.append(f"Score: {cash['score']}/100  ({cash['label']})")
    lines.append(f"Note: {cash['note']}")

    vol = result["revenue_volatility"]
    lines.append("\n--- 4. REVENUE VOLATILITY ---")
    lines.append(f"Coefficient of variation: {vol['coefficient_of_variation']}")
    lines.append(f"Score: {vol['score']}/100  ({vol['label']})")
    lines.append(f"Note: {vol['note']}")

    lines.append("\n--- ITEMS TO WATCH ---")
    if result["flags"]:
        for f in result["flags"]:
            lines.append(f"  - {f}")
    else:
        lines.append("No sub-score fell below the review threshold (40/100).")

    lines.append("\n" + "=" * 60)
    lines.append("This snapshot is formula-driven and fully auditable. Every figure above")
    lines.append("traces to the assumptions and inputs supplied. No composite score is")
    lines.append("computed by design — see the note at the top of this pipeline.")
    lines.append("Prepared via Vaelo — for review by the submitting CA before client delivery.")

    return "\n".join(lines)

## 2. Step 1 — Meridian Standalone Valuation (Pipeline 1)

In [ ]:
print("#" * 70)
print("# STEP 1 — MERIDIAN STANDALONE VALUATION (Pipeline 1)")
print("#" * 70)

meridian_valuation_request = ValuationRequest(
    meta=ClientMeta(
        client_name="Meridian Auto Components Pvt Ltd",
        ca_firm_name="Krishnan & Rao Chartered Accountants",
        trigger_event="sale",  # founder-led succession sale
        sector="Auto Components (Manufacturing)",
    ),
    financials=FinancialInputs(
        historical_revenue=[12.5, 14.8, 16.9, 18.6, 21.3],
        historical_ebitda=[1.9, 2.3, 2.7, 3.0, 3.6],
        current_ebit=2.9,
        current_da=0.7,
        current_capex=0.9,
        current_nwc_change=0.35,
        tax_rate=0.25,
        net_debt=3.2,
    ),
    wacc_inputs=WACCInputs(
        risk_free_rate=0.071,
        equity_risk_premium=0.065,
        size_premium=0.055,           # small SME — slightly above the generic example
        company_specific_premium=0.02,
        cost_of_debt=0.115,
        equity_weight=0.65,
        debt_weight=0.35,
        tax_rate=0.25,
    ),
    assumptions=ProjectionAssumptions(
        projection_years=5,
        revenue_growth_rates=[0.14, 0.12, 0.10, 0.09, 0.08],  # EV supply-chain tailwind, moderating
        ebitda_margin=0.175,
        da_as_pct_revenue=0.035,
        capex_as_pct_revenue=0.045,
        nwc_change_as_pct_revenue=0.02,
        terminal_growth_rate=0.045,
    ),
    comparables=ComparableAssumptions(
        ev_ebitda_multiple_low=5.0,
        ev_ebitda_multiple_high=6.5,
    ),
)

meridian_base_revenue = meridian_valuation_request.financials.historical_revenue[-1]
meridian_dcf = calculate_enterprise_value(meridian_base_revenue, meridian_valuation_request)

meridian_sensitivity = sensitivity_grid(
    meridian_base_revenue,
    meridian_valuation_request,
    wacc_range=[meridian_dcf["wacc"] - 0.02, meridian_dcf["wacc"], meridian_dcf["wacc"] + 0.02],
    growth_range=[
        meridian_valuation_request.assumptions.terminal_growth_rate - 0.01,
        meridian_valuation_request.assumptions.terminal_growth_rate,
        meridian_valuation_request.assumptions.terminal_growth_rate + 0.01,
    ],
)

meridian_cross_check = comparable_cross_check(
    meridian_valuation_request.financials.historical_ebitda[-1],
    meridian_valuation_request,
    meridian_dcf["enterprise_value"],
)

valuation_report = build_valuation_report(
    meridian_valuation_request, meridian_dcf, meridian_sensitivity, meridian_cross_check
)
print(valuation_report)


print("\n" + "#" * 70)
print("# STEP 2 — HORIZON INDUSTRIALS ACQUIRES MERIDIAN (Pipeline 2)")

## 3. Step 2 — Horizon Industrials Acquires Meridian (Pipeline 2)

Note the target's `standalone_value` below is set to `meridian_dcf["equity_value"]` from Step 1 — this is the handoff between pipelines.

In [ ]:
print("#" * 70)
print(f"\n[Target's standalone_value pulled directly from Pipeline 1's DCF equity")
print(f" value: Rs. {meridian_dcf['equity_value']:.2f} Cr — not re-estimated.]\n")

deal_request = DealFeasibilityRequest(
    meta=DealMeta(
        acquirer_name="Horizon Industrials Pvt Ltd",
        target_name="Meridian Auto Components Pvt Ltd",
        ca_firm_name="Krishnan & Rao Chartered Accountants",
        deal_rationale="vertical integration — Horizon expands into Tier-2 auto ancillary supply via Meridian's Coimbatore capacity",
        sector="Auto Components (Manufacturing)",
    ),
    acquirer=CompanyProfile(
        name="Horizon Industrials Pvt Ltd",
        revenue=60.0,
        ebitda=11.0,
        net_income=6.5,
        shares_outstanding=6_000_000,
        net_debt=8.0,
    ),
    target=CompanyProfile(
        name="Meridian Auto Components Pvt Ltd",
        revenue=meridian_valuation_request.financials.historical_revenue[-1],
        ebitda=meridian_valuation_request.financials.historical_ebitda[-1],
        net_income=2.0,
        shares_outstanding=800_000,
        net_debt=meridian_valuation_request.financials.net_debt,
        standalone_value=meridian_dcf["equity_value"],   # <-- Pipeline 1 -> Pipeline 2 handoff
        fallback_ev_ebitda_multiple=6.0,                  # unused since standalone_value is set
    ),
    deal_terms=DealTerms(
        deal_type="acquisition",
        purchase_price=round(meridian_dcf["equity_value"] * 1.15, 2),  # modest 15% control premium
        cash_component_pct=0.65,
        stock_component_pct=0.35,
        acquirer_share_price=750,
    ),
    financing=FinancingAssumptions(
        new_debt_raised=7.0,
        cost_of_new_debt=0.12,
        acquirer_cash_used=round(meridian_dcf["equity_value"] * 1.15 * 0.65, 2) - 7.0,
        tax_rate=0.25,
    ),
    synergies=SynergyAssumptions(
        annual_cost_synergies=0.5,
        annual_revenue_synergies=1.2,
        synergy_ebitda_margin=0.16,
        ramp_up_years=3,
        synergy_discount_rate=0.135,
    ),
)

deal_result = calculate_deal_feasibility(deal_request)
deal_report = build_deal_feasibility_report(deal_request, deal_result)
print(deal_report)


print("\n" + "#" * 70)
print("# STEP 3 — MERIDIAN FINANCIAL HEALTH SNAPSHOT (Pipeline 3)")

## 4. Step 3 — Meridian Financial Health Snapshot (Pipeline 3)

Uses the identical `historical_revenue` list from Step 1's `meridian_valuation_request`.

In [ ]:
print("#" * 70)
print("\n[Same 5-year revenue history used in Pipeline 1 — this is what makes")
print(" Meridian a credible acquisition target, not just a cheap one.]\n")

health_request = HealthSnapshotRequest(
    meta=SnapshotMeta(
        client_name="Meridian Auto Components Pvt Ltd",
        ca_firm_name="Krishnan & Rao Chartered Accountants",
        sector="Auto Components (Manufacturing)",
    ),
    liquidity=LiquidityInputs(
        current_assets=8.4,
        current_liabilities=5.1,
    ),
    expense_growth=ExpenseGrowthInputs(
        revenue_prior_year=18.6,
        revenue_current_year=21.3,
        operating_expenses_prior_year=15.6,
        operating_expenses_current_year=17.7,
    ),
    cash_runway=CashRunwayInputs(
        cash_and_equivalents=2.8,
        monthly_net_cash_flow=0.15,   # cash-generative
    ),
    revenue_volatility=RevenueVolatilityInputs(
        historical_revenue=meridian_valuation_request.financials.historical_revenue,
    ),
)

health_result = calculate_health_snapshot(health_request)
health_report = build_health_snapshot_report(health_request, health_result)
print(health_report)


print("\n" + "#" * 70)
print("# CONSISTENCY CHECK — all three reports agree with each other")

## 5. Consistency Check

Asserts the three reports actually share numbers, not just that each runs without error.

In [ ]:
print("#" * 70)
assert deal_request.target.standalone_value == meridian_dcf["equity_value"], \
    "Deal Feasibility target value must trace back to the Valuation Report's DCF equity value"
assert deal_request.target.revenue == health_request.revenue_volatility.historical_revenue[-1], \
    "Deal Feasibility target revenue must match the Health Snapshot's latest revenue year"
assert meridian_valuation_request.financials.historical_revenue == health_request.revenue_volatility.historical_revenue, \
    "Valuation and Health Snapshot must be built on the identical revenue history"
print("PASSED — target's deal value traces to the DCF, and both the deal and")
print("the health snapshot are built on the same underlying revenue history.")
print("This is one coherent example client, not three disconnected demos.")